# تحليل طلب المصاريف

يستعرض هذا الدفتر كيفية إنشاء وكلاء يستخدمون الإضافات لمعالجة مصاريف السفر من صور الإيصالات المحلية، وإنشاء بريد إلكتروني لطلب المصاريف، وتصوير بيانات المصاريف باستخدام مخطط دائري. يختار الوكلاء الوظائف ديناميكيًا بناءً على سياق المهمة.

الخطوات:
1. يقوم وكيل OCR بمعالجة صورة الإيصال المحلية واستخلاص بيانات مصاريف السفر.
2. يقوم وكيل البريد الإلكتروني بإنشاء بريد إلكتروني لطلب المصاريف.

### مثال على سيناريو مصاريف السفر:
تخيل أنك موظف يسافر لحضور اجتماع عمل في مدينة أخرى. لدى شركتك سياسة لتعويض جميع المصاريف المعقولة المتعلقة بالسفر. فيما يلي تفصيل لمصاريف السفر المحتملة:
- النقل:
تذاكر الطيران لرحلة ذهاب وعودة من مدينتك إلى مدينة الوجهة.
خدمات التاكسي أو التنقل عن طريق التطبيقات بين المطار والمنزل.
وسائل النقل المحلية في مدينة الوجهة (مثل النقل العام، سيارات الإيجار، أو التاكسي).

- الإقامة:
الإقامة في فندق لثلاث ليالٍ في فندق أعمال متوسط المستوى قريب من مكان الاجتماع.

- الوجبات:
بدل يومي للوجبات يشمل الإفطار والغداء والعشاء، استنادًا إلى سياسة الشركة لبدل اليوم.

- المصاريف المتنوعة:
رسوم مواقف السيارات في المطار.
رسوم الوصول إلى الإنترنت في الفندق.
البقشيش أو رسوم الخدمات الصغيرة.

- التوثيق:
تقدم جميع الإيصالات (الرحلات الجوية، التاكسي، الفندق، الوجبات، إلخ) ونموذجًا مكتملًا لتقرير المصاريف للسداد.


## استيراد المكتبات المطلوبة

قم باستيراد المكتبات والوحدات اللازمة للدفتر.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## تعريف نماذج المصروفات

قم بإنشاء نموذج Pydantic للمصروفات الفردية وفئة ExpenseFormatter لتحويل استعلام المستخدم إلى بيانات مصروفات منظمة.

سيتم تمثيل كل مصروف بالشكل التالي:
`{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## تعريف الأدوات - إنشاء البريد الإلكتروني

أنشئ دالة أداة لتوليد بريد إلكتروني لتقديم طلب مصروفات.
- تستخدم هذه الأداة المزخرف `@tool` من إطار عمل Microsoft Agent.
- تحسب المبلغ الإجمالي للمصروفات وتنسق التفاصيل في نص البريد الإلكتروني.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# أداة لاستخراج مصاريف السفر من صور الإيصالات

أنشئ دالة أداة لاستخراج مصاريف السفر من صور الإيصالات.  
- تستخدم هذه الأداة المزخرف `@tool` من إطار عمل مايكروسوفت إيجيانت.  
- تقرأ صورة الإيصال، ترمزها بقاعدة 64، وتُعيد URI البيانات ليحلله الوكيل.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## معالجة النفقات

قم بتعريف الوكلاء وربطهم في سير عمل تسلسلي باستخدام `WorkflowBuilder`.
- يقوم وكيل التعرف الضوئي على الحروف باستخراج بيانات النفقات المهيكلة من صورة الإيصال باستخدام أداة `load_receipt_image`.
- يأخذ وكيل البريد الإلكتروني البيانات المستخرجة وينشئ رسالة مطالبة نفقات احترافية باستخدام أداة `generate_expense_email`.
- ينشئ `WorkflowBuilder` مع `add_edge` خط أنابيب تسلسلي: وكيل OCR → وكيل البريد الإلكتروني.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## الدالة الرئيسية

قم ببناء سير العمل التسلسلي وتشغيله لمعالجة صورة الإيصال وإنشاء بريد إلكتروني لمطالبة المصروفات.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
